In [1]:
PROJECT_ID = "qwiklabs-gcp-01-5fc8b912fc6a"
DATASET    = "flights"
LOCATION   = "US"

from google.cloud import bigquery, pubsub_v1
bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

bq.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET}"), exists_ok=True)
print("dataset ready")

dataset ready


In [4]:
schema = [
    bigquery.SchemaField("MT",   "STRING",  mode="NULLABLE", description="SEL ID AIR STA CLK MSG info http://woodair.net/sbs/Article/Barebones42_Socket_Data.htm"),
    bigquery.SchemaField("TT",   "INT64",   mode="NULLABLE", description="1 - 8"),
    bigquery.SchemaField("SID",  "STRING",  mode="NULLABLE", description="Database Session record number"),
    bigquery.SchemaField("AID",  "STRING",  mode="NULLABLE", description="Database Aircraft record number"),
    bigquery.SchemaField("Hex",  "STRING",  mode="NULLABLE", description="Aircraft Mode S hexadecimal code https://opensky-network.org/datasets/metadata/"),
    bigquery.SchemaField("FID",  "STRING",  mode="NULLABLE", description="Database Flight record number"),
    bigquery.SchemaField("DMG",  "DATE",    mode="NULLABLE", description="Date message generated"),
    bigquery.SchemaField("TMG",  "TIME",    mode="NULLABLE", description="Time message generated"),
    bigquery.SchemaField("DML",  "DATE",    mode="NULLABLE", description="Date message logged"),
    bigquery.SchemaField("TML",  "TIME",    mode="NULLABLE", description="Time message logged"),
    bigquery.SchemaField("CS",   "STRING",  mode="NULLABLE", description="Callsign (flight number or registration)"),
    bigquery.SchemaField("Alt",  "INT64",   mode="NULLABLE", description="Mode C altitude (Flight Level)"),
    bigquery.SchemaField("GS",   "INT64",   mode="NULLABLE", description="Ground Speed"),
    bigquery.SchemaField("Trk",  "INT64",   mode="NULLABLE", description="Track"),
    bigquery.SchemaField("Lat",  "FLOAT64", mode="NULLABLE", description="Latitude (N/E positive, S/W negative)"),
    bigquery.SchemaField("Lng",  "FLOAT64", mode="NULLABLE", description="Longitude (N/E positive, S/W negative)"),
    bigquery.SchemaField("VR",   "INT64",   mode="NULLABLE", description="Vertical Rate"),
    bigquery.SchemaField("Sq",   "STRING",  mode="NULLABLE", description="Assigned Mode A squawk code"),
    bigquery.SchemaField("Alrt", "INT64",   mode="NULLABLE", description="Flag to indicate squawk has changed"),
    bigquery.SchemaField("Emer", "INT64",   mode="NULLABLE", description="Flag to indicate emergency code has been set"),
    bigquery.SchemaField("SPI",  "INT64",   mode="NULLABLE", description="Flag to indicate transponder Ident has been activated"),
    bigquery.SchemaField("Gnd",  "INT64",   mode="NULLABLE", description="Flag to indicate ground squat switch is active"),
]

In [5]:
subscriber = pubsub_v1.SubscriberClient()
sub_path   = subscriber.subscription_path(PROJECT_ID, "flight-sub")
topic_path = "projects/paul-leroy/topics/flight-transponder"

try:
    subscriber.create_subscription(name=sub_path, topic=topic_path)
    print("subscription created")
except Exception as e:
    print("subscription:", e)   # already exists is fine

subscription: 409 Resource already exists in the project (resource=flight-sub).


In [6]:
import time
from google.api_core import exceptions

cols  = ["MT","TT","SID","AID","Hex","FID","DMG","TMG","DML","TML","CS",
         "Alt","GS","Trk","Lat","Lng","VR","Sq","Alrt","Emer","SPI","Gnd"]
ints   = {"TT","Alt","GS","Trk","VR","Alrt","Emer","SPI","Gnd"}
floats = {"Lat","Lng"}
dates  = {"DMG","DML"}

def parse(line):
    row = {}
    for name, val in zip(cols, line.split(",")):
        val = val.strip()
        if not val:               # many fields are empty - leave them null
            continue
        if name in dates:    row[name] = val.replace("/", "-")   # date wants dashes
        elif name in ints:   row[name] = int(val)
        elif name in floats: row[name] = float(val)
        else:                row[name] = val
    return row

stop, total = time.time() + 120, 0      # collect for 2 minutes
while time.time() < stop:
    try:
        resp = subscriber.pull(subscription=sub_path, max_messages=200, timeout=10)
    except exceptions.DeadlineExceeded:
        continue
    if not resp.received_messages:
        continue
    rows = [parse(m.message.data.decode()) for m in resp.received_messages]
    errors = bq.insert_rows_json(table_id, rows)
    if errors:
        print("insert errors:", errors[:2])
    subscriber.acknowledge(subscription=sub_path,
                           ack_ids=[m.ack_id for m in resp.received_messages])
    total += len(rows)
    print("inserted", total)

print("done, total:", total)

inserted 200
inserted 400
inserted 600
inserted 800
inserted 1000
inserted 1200
inserted 1400
inserted 1600
inserted 1800
inserted 2000
inserted 2200
inserted 2400
inserted 2600
inserted 2800
inserted 3000
inserted 3200
inserted 3400
inserted 3600
inserted 3800
inserted 4000
inserted 4200
inserted 4400
inserted 4600
inserted 4800
inserted 5000
inserted 5200
inserted 5400
inserted 5600
inserted 5800
inserted 6000
inserted 6200
inserted 6400
inserted 6600
inserted 6800
inserted 7000
inserted 7058
inserted 7258
inserted 7458
inserted 7658
inserted 7858
inserted 8058
inserted 8258
inserted 8458
inserted 8658
inserted 8858
inserted 9058
inserted 9228
inserted 9428
inserted 9628
inserted 9828
inserted 10028
inserted 10228
inserted 10428
inserted 10628
inserted 10828
inserted 11028
inserted 11228
inserted 11428
inserted 11628
inserted 11828
inserted 12028
inserted 12228
inserted 12428
inserted 12628
inserted 12828
inserted 13028
inserted 13228
inserted 13428
inserted 13628
inserted 13828
inse

In [7]:
bq.query(f"SELECT COUNT(*) AS records FROM `{table_id}`").to_dataframe()

,records
0,41683


In [8]:
geoviz_sql = f"""
SELECT ST_GEOGPOINT(Lng, Lat) AS location
FROM `{table_id}`
WHERE Lat IS NOT NULL AND Lng IS NOT NULL
"""
print(geoviz_sql)


SELECT ST_GEOGPOINT(Lng, Lat) AS location
FROM `qwiklabs-gcp-01-5fc8b912fc6a.flights.messages`
WHERE Lat IS NOT NULL AND Lng IS NOT NULL

